# Minimal Training Test (~1 min)

Just verifies training works and losses decrease.

In [1]:
import torch
import lightning as L
from pathlib import Path
from ase.build import bulk
from ase.io import write

from NPS.logp.models import LitLogPModel
from NPS.logp.data import PeriodicStructureDataModule

/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


In [2]:
# Create tiny dataset
data_dir = Path("tiny_data")
data_dir.mkdir(exist_ok=True)

write(data_dir / 'bcc.extxyz', bulk('Fe', 'bcc', a=2.87, cubic=True) * (3,3,3))
write(data_dir / 'fcc.extxyz', bulk('Cu', 'fcc', a=3.61, cubic=True) * (3,3,3))

structure_types = ["bcc", "fcc"]

In [3]:
# DataModule
dm = PeriodicStructureDataModule(
    file_list=[str(data_dir / f"{s}.extxyz") for s in structure_types],
    cutoff=4.0,
    duplicate=20,
    batch_size=2,
    num_workers=0,
    structure_types=structure_types,
)

# Tiny model
model = LitLogPModel(
    nclass=2,
    cutoff=4.0,
    hidden_irreps="16x0e+16x1o+16x2e",
    num_interactions=1,
    sigma_max=0.15,
    sigma_logp_scale=0.15,
    learn_rate=1e-3,
)

print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/mace/modules/blocks.py:312: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(atomic_energies, dtype=torch.get_default_dtype()),
/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.

Params: 531,296


/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(


In [4]:
# Train
trainer = L.Trainer(accelerator='cpu', max_epochs=3, enable_checkpointing=False, logger=False)
trainer.fit(model, dm)
print("Done!")

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.

  | Name      | Type             | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | model     | LogPModelWrapper | 265 K  | train | 0    
1 | ema_model | AveragedModel    | 265 K  | train | 0    
---------------------------------------------------------------
531 K     Trainable params
0         Non-trainable params
531 K     Total params
2.125     Total estimated model params size (MB)
143       Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |                                                                                            …

/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 216. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we f

Training: |                                                                                                   …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

`Trainer.fit` stopped: `max_epochs=3` reached.


Done!
